# FetalScan on Colab
Runtime > Change runtime type > **T4 GPU**. Runs the full pipeline on the real public datasets (about 1-2 hours total).

In [ ]:
!git clone https://github.com/churnika67/fetalscan.git
%cd fetalscan
!pip -q install -r requirements.txt

In [ ]:
!bash scripts/download_data.sh

## 1. Plane classification (FETAL_PLANES_DB, ResNet-18)

In [ ]:
!python -m fetalscan.train_cls --data data/FETAL_PLANES_ZENODO --epochs 15 --batch-size 64 --workers 2 --out runs/cls --mlflow

In [ ]:
from IPython.display import Image, display
import json
print(json.dumps(json.load(open('runs/cls/metrics.json'))['test'], indent=1)[:600])
display(Image('runs/cls/confusion_matrix.png'))

## 2. Optional: Optuna hyperparameter search (short runs on a subset)

In [ ]:
!python -m fetalscan.tune --data data/FETAL_PLANES_ZENODO --trials 10 --epochs 3 --limit 400

## 3. Head segmentation + head circumference (HC18, U-Net + OpenCV ellipse fit)

In [ ]:
!python -m fetalscan.train_seg --data data/HC18 --epochs 40 --batch-size 8 --workers 2 --out runs/seg --mlflow

In [ ]:
import glob
print(json.load(open('runs/seg/metrics.json'))['test'])
for p in sorted(glob.glob('runs/seg/examples/*.png'))[:4]: display(Image(p))

## 4. Grad-CAM: where does the classifier look?

In [ ]:
import pandas as pd
from fetalscan.datasets import read_planes_csv
df = read_planes_csv('data/FETAL_PLANES_ZENODO')
paths = df[df.Train==0].groupby('Plane').head(1).path.tolist()
!python -m fetalscan.explain --ckpt runs/cls/best.pt --images {' '.join(paths)} --out runs/gradcam
for p in sorted(glob.glob('runs/gradcam/*.png')): display(Image(p))

## 5. ONNX export + real-time latency

In [ ]:
!python -m fetalscan.export --cls runs/cls/best.pt --seg runs/seg/best.pt --out exports
print(open('exports/latency.json').read())

## 6. Save results
Download `runs/*/metrics.json`, `exports/latency.json`, the confusion matrix, Grad-CAM and segmentation examples. Paste the metrics back into the README results table.

In [ ]:
!zip -qr results.zip runs/cls/metrics.json runs/cls/confusion_matrix.png runs/seg/metrics.json runs/seg/examples runs/gradcam exports/latency.json runs/tune/study.json
from google.colab import files; files.download('results.zip')